In [1]:
def psi(n,x,y,A,B,F):
    if n==0:return F(0)
    if n==1:return F(1)
    if n==2:return F(2)*y
    if n==3:return F(3)*x^4+F(6)*A*x^2+F(12)*B*x-A^2
    if n==4:return F(4)*y*(x^6+F(5)*A*x^4+F(20)*B*x^3-F(5)*A^2*x^2-F(4)*A*B*x-F(8)*B^2-A^3)
    m={i:psi(i,x,y,A,B,F) for i in range(5)}
    def f(n):
        if n in m:return m[n]
        if n&1:
            k=(n-1)//2
            v=f(k+2)*f(k)^3-f(k-1)*f(k+1)^3
        else:
            k=n//2
            v=f(k)*(f(k+2)*f(k-1)^2-f(k-2)*f(k+1)^2)/(F(2)*y)
        m[n]=v
        return v
    return f(n)

def psi_wrap(n,x,y,A,B,F):
    return (-1 if n<0 else 1)*psi(abs(n),x,y,A,B,F)

def add(P,Q,A,F):
    if P==(None,None):return Q
    if Q==(None,None):return P
    x1,y1=P;x2,y2=Q
    if x1==x2 and y1==-y2:return None,None
    lam=(F(3)*x1^2+A)/(F(2)*y1) if P==Q else (y2-y1)/(x2-x1)
    x3=lam^2-x1-x2
    return x3,lam*(x1-x3)-y1

def scalar_mult(k,P,A,F):
    R=(None,None)
    for b in bin(k)[2:]:
        R=add(R,R,A,F)
        if b=='1':R=add(R,P,A,F)
    return R

def Theorem5(P,k,n,A,B,F):
    """Test ψ_n(kP) * ψ_k(P)^{n^2} = ψ_{n*k}(P) mod p"""
    xP,yP=P.xy()
    kP=scalar_mult(k,(xP,yP),A,F)
    psi_n_kP=psi(n,*kP,A,B,F) if kP!=(None,None) else F(0)
    psi_k_P=psi(k,xP,yP,A,B,F)
    lhs=psi_n_kP*psi_k_P^(n^2)
    rhs=psi(n*k,xP,yP,A,B,F)
    print(f"k={k}, kP={kP}")
    print(f"ψ_n(kP)={psi_n_kP}")
    print(f"ψ_k(P)={psi_k_P}")
    print(f"LHS={lhs}")
    print(f"RHS={rhs}")
    print(f"Match={lhs==rhs}")
    return lhs==rhs

def FindABC(xP,yP, M, A, B, p,F):
    #Find a,b,c
    psi_M_minus1_P = psi(M - 1, xP, yP, A, B,F)
    psi_M_minus2_P = psi(M - 2, xP, yP, A, B,F)
    psi_2_P = psi(2, xP, yP, A, B,F)
    a = (psi_M_minus2_P * pow(psi_M_minus1_P * psi_2_P,-1,p))
    b = (psi_M_minus1_P * psi_M_minus1_P * psi_2_P * pow(psi_M_minus2_P,-1,p))
    c = (psi_M_minus1_P * psi_2_P * pow(psi_M_minus2_P,-1,p))
    return a,b,c

def TestSuperSingularCurve():
    p=21616
    while p%3!=2:p=next_prime(p)
    A=0;B=11;F=GF(p)
    E=EllipticCurve(F,[A,B])
    P=E.gens()[0]
    print(f"p={p}\nE={E}\n#E={E.cardinality()}\nP={P}\nord(P)={P.order()}")
    return E,P,E.cardinality(),p,A,B,F

#ellipticCurve,P,order,fieldCharacteristic,A,B,F=TestSuperSingularCurve()


In [12]:
def FindKoblitzCurve_PrimeOrder(B, primeStart=5):
    p = next_prime(primeStart - 1) 
    while True:
        if B % p != 0 and (4*B^3 + 27) % p != 0:
            E = EllipticCurve(GF(p), [0, B])
            N = E.order()
            if N.is_prime() and N != p:
                return E
        p = next_prime(p)
        

primeStart=2094547
A = 0;B = 7
E = FindKoblitzCurve_PrimeOrder(B,primeStart);
primeNumber = E.base_field().characteristic();F=GF(primeNumber)  
pointOrder  = E.order()                         
G           = E.gens()[0]                     
generatorX  = G[0]
generatorY  = G[1]
a,b,c=FindABC(G[0],G[1],pointOrder,A,B,primeNumber,F)
cOrder=c.multiplicative_order()
factors=factor(cOrder)
print(f"factors = {factors}")
print(f" primeNumber = {primeNumber}")
print(f" pointOrder  = {pointOrder}")
print(f" generatorX  = {generatorX}")
print(f" generatorY  = {generatorY}")
print(f" ell0  = {factors[0][0]}")
print(f" ell1  = {factors[1][0]}")

factors = 3 * 19 * 9187
 primeNumber = 2094637
 pointOrder  = 2093317
 generatorX  = 61215
 generatorY  = 1573888
 ell0  = 3
 ell1  = 19
